# G1 step-length training (Colab)
Generated by `scripts/make_colab.py`. Edit `g1pipe/` in the repo, not this notebook.

1. Runtime → Change runtime type → **T4 GPU**
2. Set `TIMESTEPS` / `RUN_NAME` below
3. Run all. Checkpoints are written to Google Drive every eval, so a disconnect loses at most one eval interval.

In [ ]:
TIMESTEPS = 200_000_000   # 5_000_000 for a quick probe
RUN_NAME = 'g1-stairs-v4'
EXTRA_ARGS = ['--task', 'stairs', '--scan-model', 'camera']   # [] for flat step length; ['--no-dr'] for E3
INIT_FROM = 'g1pipe_init/g1-stairs-v3.pkl'   # params.pkl to warm-start from, relative to MyDrive; '' for none
USE_DRIVE = True

In [ ]:
!nvidia-smi
!pip install -q 'playground==0.2.0' 'brax==0.14.2' 'jax[cuda12]==0.7.2' 'flax==0.12.0' 'mujoco==3.14.0' 'mujoco-mjx==3.14.0' 'warp-lang==1.17.0'

In [ ]:
import pathlib
FILES = {'__init__.py': '', 'steplength_env.py': '"""G1 step-length task: MuJoCo Playground\'s G1 joystick env + explicit step-length control.\n\nOperator command: forward speed vx and step length l*. Cadence follows from them:\n    gait_freq f = |vx| / (2 l*)      (two steps per gait cycle)\nso during training we sample (vx, f) over a wide range and give the policy both\nf and l* = vx / (2 f) as observations. A touchdown reward pays for landing the\nswing foot l* ahead of the stance foot along the heading.\n\nChanges vs. upstream G1 Joystick (kept deliberately small, see docs/pipeline.md):\n  * gait_freq range widened 1.25–1.5 Hz  ->  GAIT_FREQ_RANGE\n  * obs "state" gets [l*, f] appended (+2)\n  * new reward term "step_length"\n"""\nfrom __future__ import annotations\n\nimport jax\nimport jax.numpy as jp\nfrom ml_collections import config_dict\nfrom mujoco_playground._src import mjx_env\nfrom mujoco_playground._src.locomotion.g1 import joystick as g1_joystick\n\nmjx_env.ensure_menagerie_exists()  # downloads G1 meshes on first use\n\nGAIT_FREQ_RANGE = (0.9, 1.8)      # Hz  -> gait period 0.56–1.11 s\nSTEP_SIGMA = 0.05                 # m, width of touchdown reward\nMIN_WALK_SPEED = 0.15             # m/s, below this the step reward is off\n\n\ndef default_config() -> config_dict.ConfigDict:\n    cfg = g1_joystick.default_config()\n    cfg.reward_config.scales.step_length = 1.5\n    cfg.lin_vel_x = [-0.5, 1.2]\n    return cfg\n\n\nclass StepLength(g1_joystick.Joystick):\n\n    def __init__(self, task: str = "flat_terrain", config=None, config_overrides=None):\n        super().__init__(task=task, config=config or default_config(), config_overrides=config_overrides)\n\n    # -- command ---------------------------------------------------------------\n    def _sample_gait(self, rng, command):\n        f = jax.random.uniform(rng, (), minval=GAIT_FREQ_RANGE[0], maxval=GAIT_FREQ_RANGE[1])\n        step_cmd = command[0] / (2.0 * f)\n        return f, step_cmd\n\n    def _set_gait(self, info, rng):\n        f, step_cmd = self._sample_gait(rng, info["command"])\n        info["gait_freq"] = f\n        info["step_cmd"] = step_cmd\n        info["phase_dt"] = 2 * jp.pi * self.dt * jp.array([f])\n        return info\n\n    # -- env API ---------------------------------------------------------------\n    def reset(self, rng):\n        rng, gait_rng = jax.random.split(rng)\n        state = super().reset(rng)\n        info = self._set_gait(dict(state.info), gait_rng)\n        state.metrics["step_len_err"] = jp.zeros(())\n        contact = self._contact(state.data)\n        obs = self._get_obs(state.data, info, contact)\n        return state.replace(obs=obs, info=info)\n\n    def step(self, state, action):\n        state = super().step(state, action)\n        # Upstream resamples the velocity command when info["step"] wraps to 0; resample cadence too.\n        info = dict(state.info)\n        info["rng"], gait_rng = jax.random.split(info["rng"])\n        new = self._set_gait(dict(info), gait_rng)\n        wrapped = info["step"] == 0\n        for k in ("gait_freq", "step_cmd", "phase_dt"):\n            info[k] = jp.where(wrapped, new[k], info[k])\n        return state.replace(info=info)\n\n    def _contact(self, data):\n        return jp.array([\n            data.sensordata[self._mj_model.sensor_adr[s]] > 0 for s in self._feet_floor_found_sensor\n        ])\n\n    def _get_obs(self, data, info, contact):\n        obs = super()._get_obs(data, info, contact)\n        extra = jp.hstack([\n            info.get("step_cmd", jp.zeros(())) * 4.0,          # ~[-1, 1] scale\n            info.get("gait_freq", jp.array(1.35)) - 1.35,\n        ])\n        return {\n            "state": jp.hstack([obs["state"], extra]),\n            "privileged_state": jp.hstack([obs["privileged_state"], extra]),\n        }\n\n    def _get_reward(self, data, action, info, metrics, done, first_contact, contact):\n        rewards = super()._get_reward(data, action, info, metrics, done, first_contact, contact)\n        rewards["step_length"] = self._reward_step_length(data, info, first_contact, metrics)\n        return rewards\n\n    def _reward_step_length(self, data, info, first_contact, metrics):\n        feet_xy = data.site_xpos[self._feet_site_id][:, :2]\n        fwd = data.xmat[self._torso_body_id][:2, 0]\n        fwd = fwd / (jp.linalg.norm(fwd) + 1e-6)\n        # step length of foot i = how far it landed ahead of the other foot\n        step = jp.array([\n            jp.dot(feet_xy[0] - feet_xy[1], fwd),\n            jp.dot(feet_xy[1] - feet_xy[0], fwd),\n        ])\n        err = step - info.get("step_cmd", jp.zeros(()))\n        per_foot = jp.exp(-jp.square(err / STEP_SIGMA)) * first_contact\n        walking = jp.abs(info["command"][0]) > MIN_WALK_SPEED\n        # metric: abs error on touchdown steps (0 when no touchdown this tick)\n        n_td = jp.maximum(jp.sum(first_contact), 1.0)\n        metrics["step_len_err"] = jp.sum(jp.abs(err) * first_contact) / n_td * walking\n        return jp.sum(per_foot) * walking\n', 'stairs_terrain.py': '"""Stairs terrain for training and evaluating a stair-climbing G1 policy.\n\nA GRID x GRID grid of CELL x CELL m staircases (legged_gym-style terrain curriculum):\n\n  * row iy = difficulty level: every staircase in a row has the same step height\n  * columns alternate between a pyramid (steps up to a central platform) and a pit\n    (steps down to a central floor), so a robot spawned in the middle of a cell walks\n    down stairs (pyramid) or up stairs (pit) whichever way it heads\n  * a flat MARGIN around the grid; flat ground is at z = 0, pits go below it\n\nTwo layouts: "train" (the curriculum) and "test" (held out: step heights between the\ntraining levels and a different tread depth), so evaluation never replays training stairs.\n\nOne heightfield named "floor" holds it all, so the G1 model\'s foot-floor contact pairs and\nsensors work unchanged, in MJX/Warp and in plain MuJoCo. The same grid gives the height scan:\na patch of points around the robot, ground height looked up bilinearly (numpy or jax),\nrelative to the robot\'s nominal foot level. On hardware this comes from the head depth\ncamera / LiDAR elevation map (see jev_agent/vision_check.py).\n"""\nfrom __future__ import annotations\n\nimport numpy as np\n\nCELL = 4.0            # m\nGRID = 8              # cells per side; also the number of curriculum levels\nMARGIN = 2.0          # m of flat ground around the grid\nRES = 0.05            # m per heightfield sample\nPLATFORM = 1.0        # m, flat centre of each pyramid / pit\nBORDER = 0.35         # m, flat strip around each cell (walkway between staircases)\n\nLAYOUTS = {\n    "train": {"rises": np.linspace(0.02, 0.16, GRID), "tread": 0.30},   # 2, 4, ... 16 cm\n    "test": {"rises": np.linspace(0.03, 0.15, GRID), "tread": 0.27},    # 3.0, 4.7, ... 15 cm\n}\n\n# height scan pattern, robot yaw frame: 11 x 5 points from 0.3 m behind to 1.2 m ahead\nSCAN_X = np.linspace(-0.3, 1.2, 11)\nSCAN_Y = np.linspace(-0.3, 0.3, 5)\nSCAN_PTS = np.stack(np.meshgrid(SCAN_X, SCAN_Y, indexing="ij"), -1).reshape(-1, 2)\nN_SCAN = len(SCAN_PTS)\nNOMINAL_HEIGHT = 0.755   # pelvis height above the feet when standing (knees_bent keyframe)\n\nGRID_HALF = GRID * CELL / 2\nHALF = GRID_HALF + MARGIN\nN = int(round(2 * HALF / RES)) + 1\n\n\ndef n_steps(layout: str = "train") -> int:\n    """Steps per flight, from the cell border to the central platform."""\n    return int((CELL / 2 - BORDER - PLATFORM / 2) // LAYOUTS[layout]["tread"])\n\n\ndef cell_rises(layout: str = "train") -> np.ndarray:\n    """Step height of each cell, indexed [iy, ix]."""\n    return np.repeat(LAYOUTS[layout]["rises"][:, None], GRID, axis=1)\n\n\ndef cell_kinds(layout: str = "train") -> np.ndarray:\n    """+1 for a pyramid (centre raised), -1 for a pit (centre sunk), indexed [iy, ix]."""\n    return np.tile(np.where(np.arange(GRID) % 2 == 0, 1, -1), (GRID, 1))\n\n\ndef cell_center(ix, iy, xp=np):\n    return xp.stack([-GRID_HALF + (ix + 0.5) * CELL, -GRID_HALF + (iy + 0.5) * CELL], -1)\n\n\ndef heights(layout: str = "train") -> np.ndarray:\n    """Ground height grid (world z), shape (N, N), indexed [iy, ix]; x, y from -HALF to +HALF."""\n    xs = np.linspace(-HALF, HALF, N)\n    X, Y = np.meshgrid(xs, xs, indexing="xy")          # [iy, ix]\n    inside = (np.abs(X) < GRID_HALF) & (np.abs(Y) < GRID_HALF)\n    cx = np.clip(((X + GRID_HALF) // CELL).astype(int), 0, GRID - 1)\n    cy = np.clip(((Y + GRID_HALF) // CELL).astype(int), 0, GRID - 1)\n    lx, ly = X + GRID_HALF - cx * CELL, Y + GRID_HALF - cy * CELL  # position within the cell\n    d_edge = np.minimum(np.minimum(lx, CELL - lx), np.minimum(ly, CELL - ly)) - BORDER\n    steps = np.clip(np.floor(d_edge / LAYOUTS[layout]["tread"]) + 1, 0, n_steps(layout)) * (d_edge >= 0)\n    return steps * cell_rises(layout)[cy, cx] * cell_kinds(layout)[cy, cx] * inside\n\n\ndef scene_xml(layout: str = "train") -> str:\n    """MJCF scene: Playground\'s feet-only G1 + sensors on the stairs heightfield."""\n    h = heights(layout)\n    lo, hi = float(h.min()), float(h.max())\n    span = max(hi - lo, 1e-3)\n    # MJCF inline elevation lists rows from +y down to -y; our grid is indexed from -y up.\n    # MuJoCo scales elevation to [0, 1] * size_z, so the geom is shifted down to the pit floor.\n    elev = " ".join(f"{v:.5f}" for v in ((h[::-1] - lo) / span).ravel())\n    return f"""<mujoco model="g1 stairs">\n  <include file="g1_mjx_feetonly.xml"/>\n  <statistic center="0 0 0.7" extent="1.2" meansize="0.04"/>\n  <visual>\n    <headlight diffuse=".8 .8 .8" ambient=".2 .2 .2" specular="1 1 1"/>\n    <global azimuth="120" elevation="-20"/>\n    <quality shadowsize="8192"/>\n  </visual>\n  <asset>\n    <texture type="skybox" builtin="gradient" rgb1="1 1 1" rgb2="1 1 1" width="800" height="800"/>\n    <texture type="2d" name="groundplane" builtin="checker" mark="edge" rgb1=".85 .82 .76" rgb2=".78 .75 .7"\n      markrgb=".3 .3 .3" width="300" height="300"/>\n    <material name="groundplane" texture="groundplane" texuniform="true" texrepeat="40 40" reflectance="0"/>\n    <hfield name="stairs" nrow="{N}" ncol="{N}" size="{HALF} {HALF} {span:.5f} 0.1" elevation="{elev}"/>\n  </asset>\n  <worldbody>\n    <geom name="floor" type="hfield" hfield="stairs" pos="0 0 {lo:.5f}" material="groundplane"/>\n  </worldbody>\n  <include file="sensor.xml"/>\n  <keyframe>\n    <key name="knees_bent"\n      qpos="0 0 0.755 1 0 0 0\n      -0.312 0 0 0.669 -0.363 0 -0.312 0 0 0.669 -0.363 0 0 0 0.073\n      0.2 0.2 0 0.6 0 0 0 0.2 -0.2 0 0.6 0 0 0"\n      ctrl="-0.312 0 0 0.669 -0.363 0 -0.312 0 0 0.669 -0.363 0 0 0 0.073\n      0.2 0.2 0 0.6 0 0 0 0.2 -0.2 0 0.6 0 0 0"/>\n  </keyframe>\n</mujoco>\n"""\n\n\ndef lookup(grid, xy, xp=np):\n    """Bilinear ground height at points xy (..., 2). Works with numpy or jax.numpy as xp."""\n    f = (xy + HALF) / RES\n    f = xp.clip(f, 0.0, N - 1.001)\n    i0 = xp.floor(f).astype(int)\n    w = f - i0\n    ix, iy = i0[..., 0], i0[..., 1]\n    wx, wy = w[..., 0], w[..., 1]\n    h00, h01 = grid[iy, ix], grid[iy, ix + 1]\n    h10, h11 = grid[iy + 1, ix], grid[iy + 1, ix + 1]\n    return (h00 * (1 - wx) + h01 * wx) * (1 - wy) + (h10 * (1 - wx) + h11 * wx) * wy\n\n\ndef scan(grid, base_xyz, yaw, xp=np):\n    """Height scan: ground height at SCAN_PTS (robot yaw frame) relative to nominal foot level."""\n    c, s = xp.cos(yaw), xp.sin(yaw)\n    pts = xp.asarray(SCAN_PTS)\n    world = base_xyz[:2] + xp.stack([c * pts[:, 0] - s * pts[:, 1], s * pts[:, 0] + c * pts[:, 1]], -1)\n    return xp.clip(lookup(grid, world, xp) - (base_xyz[2] - NOMINAL_HEIGHT), -1.0, 1.0)\n\n\ndef scan_stats(layout: str = "train", n: int = 20000, seed: int = 0):\n    """Mean and std of each scan point over random standing poses on the grid, for\n    initialising the observation normaliser when warm-starting from a flat policy."""\n    rng = np.random.default_rng(seed)\n    grid = heights(layout)\n    xy = rng.uniform(-GRID_HALF, GRID_HALF, (n, 2))\n    yaw = rng.uniform(-np.pi, np.pi, n)\n    z = lookup(grid, xy) + NOMINAL_HEIGHT\n    s = np.stack([scan(grid, np.array([*p, h]), a) for p, h, a in zip(xy, z, yaw)])\n    return s.mean(0), s.std(0) + 1e-3\n', 'stairs_env.py': '"""G1 step-length task on stairs, with a height scan and a terrain curriculum.\n\nSame task, rewards and PPO recipe as g1pipe.steplength_env.StepLength (Playground G1\njoystick + step-length command), plus what a policy needs to climb stairs:\n\n  * terrain: g1pipe.stairs_terrain curriculum grid (row = level, 2 -> 16 cm steps;\n    pyramids to walk down, pits to climb out of)\n  * observation += 55-point height scan (11 x 5 patch, 0.3 m behind to 1.2 m ahead)\n  * feet_phase reward measures swing-foot height above the ground under the foot,\n    not above z = 0 (otherwise standing on a step looks like a raised foot)\n  * terrain curriculum (legged_gym): each env has a level and spawns in the middle of a\n    staircase of that level. At the end of an episode it moves up a level if it walked\n    off its staircase (> PROMOTE_DIST from spawn), down if it fell or covered less than\n    half the commanded distance. Envs that pass the top level get a random level, so\n    easy terrain is not forgotten.\n  * walking off the map ends the episode but is not punished as a fall\n\nv3 (stable climbing; v2 stalled at the first riser and crouched or sat on the steps):\n\n  * spawn either in the middle of a staircase facing out, or on its border facing in, always\n    square to the steps (+-YAW_JITTER): half the episodes now start with a climb\n  * commands mostly forward (STAIRS_VX), little sideways walking or turning; stairs are\n    crossed head on, the way a supervisor would command them\n  * a fall is also a torso tilted past MAX_TILT_DEG or a pelvis lower than MIN_PELVIS_REL\n    above the ground (feet-only collisions let a collapsed robot sink through the steps\n    without ever tipping 90 deg, so v2 was never punished for sitting down)\n  * feet_phase clearance is measured against the highest ground under the foot or up to\n    TOE_LOOKAHEAD ahead of it, so the swing foot lifts before it reaches a riser\n  * feet_edge: cost while a loaded foot straddles a step edge (heel and toe on different\n    steps); base_height_rel: cost for the pelvis sinking below BASE_HEIGHT_REL above the ground\n  * step_length reward off: on stairs the tread sets where the feet land\n\nscan_model = "camera" (config): the policy\'s height scan gets the errors of the simulated head\ndepth camera + elevation map (jev_agent.vision), measured in plain MuJoCo on the held-out stairs\n(results/stairs/camera_scan_v2.json): ~1 cm noise, rare 5 cm outliers at step edges, points near\nand behind the feet not yet seen (read as level with the feet), plus a per-episode map offset for\nodometry drift, which the simulated camera (perfect pose) cannot show. The critic keeps the true scan.\n\nThe curriculum lives in state.info["curriculum"]. Playground\'s auto-reset wrapper\n(full_reset=False) restores data and obs on done but keeps info, and every episode\nstarts from the cached reset data at time 0, so step() re-places the robot on the first\nstep of each episode. Train with num_resets_per_eval = 0, or host-side resets would wipe\nthe levels (g1pipe.train does this).\n\n    python -m g1pipe.train --task stairs --init-from runs/g1-stairs-v2/run/params.pkl \\\n        --timesteps 200_000_000 --out runs/stairs_v3\n"""\nfrom __future__ import annotations\n\nimport tempfile\nfrom pathlib import Path\n\nimport jax\nimport jax.numpy as jp\nfrom mujoco import mjx\nfrom mujoco_playground._src import gait\nfrom mujoco_playground._src.locomotion.g1 import base as g1_base\n\nfrom g1pipe import stairs_terrain as T\nfrom g1pipe.steplength_env import StepLength, default_config as flat_config\n\nSCAN_NOISE = 0.02        # m, uniform noise on the height scan\nN_LEVELS = T.GRID\nMAX_INIT_LEVEL = 2       # training envs start on 2-6 cm steps\nSPAWN_JITTER = 0.3       # m around the cell centre (platform half-width is 0.5 m)\nPROMOTE_DIST = T.CELL / 2 - 0.1   # m from spawn: past the last step, on the flat walkway\nDEMOTE_FRAC = 0.5        # fraction of the commanded distance an episode must cover\nEDGE_SPAWN_P = 0.5       # share of episodes that start on a cell border, facing the stairs\nYAW_JITTER = 0.3         # rad around square to the steps\nSTAIRS_VX = [0.4, 0.9]   # m/s forward command (v2: momentum helped; 0.3 m/s stalled)\nMAX_TILT_DEG = 60.0      # torso tilt that counts as a fall\nMIN_PELVIS_REL = 0.45    # m, pelvis above the ground under it that counts as a fall (stand ~0.75)\nBASE_HEIGHT_REL = 0.70   # m, pelvis height below which base_height_rel starts to cost\nTOE_LOOKAHEAD = (0.08, 0.16)   # m ahead of the foot site checked for a riser (toe is at +0.13)\nHEEL, TOE = -0.05, 0.13  # m, foot box extent along the foot from the foot site\nSOLE = 0.037             # m, foot box bottom below the foot site\n\n# camera scan model (scan_model="camera"), fitted to results/stairs/camera_scan_v2.json\nCAM_SIGMA = 0.01                  # m, noise on seen points (measured std 0.9-1.3 cm)\nCAM_OUTLIER_P, CAM_OUTLIER_SIGMA = 0.01, 0.05   # edge outliers (measured p99 5.2 cm)\nCAM_BLIND_P = {-0.3: 0.21, -0.15: 0.18, 0.0: 0.15, 0.15: 0.11, 0.3: 0.05, 0.45: 0.01}   # unseen share by row\nCAM_DRIFT_XY, CAM_DRIFT_Z = 0.03, 0.02          # m, per-episode map offset (odometry drift)\n\n\ndef default_config():\n    cfg = flat_config()\n    cfg.lin_vel_x = STAIRS_VX\n    cfg.lin_vel_y = [-0.15, 0.15]\n    cfg.ang_vel_yaw = [-0.4, 0.4]\n    cfg.reward_config.max_foot_height = 0.18\n    cfg.reward_config.scales.step_length = 0.0   # stairs set their own step length\n    cfg.reward_config.scales.feet_edge = -1.0\n    cfg.reward_config.scales.base_height_rel = -20.0\n    cfg.scan_model = "uniform"   # or "camera"\n    return cfg\n\n\nclass StairsStepLength(StepLength):\n\n    def __init__(self, config=None, config_overrides=None, layout: str = "train", eval_levels: bool = False):\n        """eval_levels: start every episode on a uniformly random level (for the PPO eval env),\n        instead of the curriculum\'s easy start."""\n        xml = Path(tempfile.gettempdir()) / f"g1_stairs_{layout}.xml"\n        xml.write_text(T.scene_xml(layout))\n        # StepLength/Joystick pick the XML from a task name; go straight to the G1 base instead.\n        g1_base.G1Env.__init__(self, xml_path=xml.as_posix(), config=config or default_config(),\n                               config_overrides=config_overrides)\n        self._post_init()\n        self._hgrid = jp.asarray(T.heights(layout), dtype=jp.float32)\n        self._eval_levels = eval_levels\n\n    # -- terrain helpers -------------------------------------------------------------\n    def _yaw(self, data):\n        q = data.qpos[3:7]\n        return jp.arctan2(2 * (q[0] * q[3] + q[1] * q[2]), 1 - 2 * (q[2] ** 2 + q[3] ** 2))\n\n    def _scan(self, data):\n        return T.scan(self._hgrid, data.qpos[:3], self._yaw(data), xp=jp)\n\n    def _place(self, data, rng, level):\n        """Robot on a random staircase of `level`, square to its steps: in the middle facing out\n        (walks down a pyramid, up out of a pit) or on the border facing in (up a pyramid, down\n        into a pit). Playground\'s joint and velocity perturbations. Returns (data, spawn xy)."""\n        k_ix, k_xy, k_yaw, k_q, k_v, k_dir, k_edge = jax.random.split(rng, 7)\n        ix = jax.random.randint(k_ix, (), 0, T.GRID)\n        ang = jax.random.randint(k_dir, (), 0, 4) * (jp.pi / 2)\n        out = jp.array([jp.cos(ang), jp.sin(ang)])\n        edge = jax.random.bernoulli(k_edge, EDGE_SPAWN_P)\n        jitter = jax.random.uniform(k_xy, (2,), minval=-SPAWN_JITTER, maxval=SPAWN_JITTER)\n        # on the border strip: along the axis stay on the strip, across it anywhere near the middle\n        along = jp.where(edge, T.CELL / 2 - T.BORDER / 2 + jitter[0] / 3, jitter[0])\n        across = jitter[1] * jp.where(edge, 1.5, 1.0)\n        xy = T.cell_center(ix, level, xp=jp) + along * out + across * jp.array([-out[1], out[0]])\n        # highest ground under the footprint, so a spawn near an edge never puts a foot inside a step\n        ring = xy + jp.array([[0, 0], [0.2, 0], [-0.2, 0], [0, 0.2], [0, -0.2], [0.2, 0.2], [-0.2, -0.2], [0.2, -0.2], [-0.2, 0.2]])\n        ground = jp.max(T.lookup(self._hgrid, ring, xp=jp))\n        yaw = ang + jp.where(edge, jp.pi, 0.0) + jax.random.uniform(k_yaw, (), minval=-YAW_JITTER, maxval=YAW_JITTER)\n        qpos = (self._init_q\n                .at[0:2].set(xy).at[2].set(ground + self._init_q[2] + 0.02)\n                .at[3:7].set(jp.array([jp.cos(yaw / 2), 0, 0, jp.sin(yaw / 2)]))\n                .at[7:].multiply(jax.random.uniform(k_q, (self._init_q.shape[0] - 7,), minval=0.5, maxval=1.5)))\n        qvel = jp.zeros_like(data.qvel).at[0:6].set(jax.random.uniform(k_v, (6,), minval=-0.5, maxval=0.5))\n        return data.replace(qpos=qpos, qvel=qvel), xy\n\n    # -- env API ---------------------------------------------------------------------\n    def reset(self, rng):\n        rng, lvl_rng, place_rng, drift_rng = jax.random.split(rng, 4)\n        state = super().reset(rng)\n        top = N_LEVELS if self._eval_levels else MAX_INIT_LEVEL + 1\n        level = jax.random.randint(lvl_rng, (), 0, top)\n        data, xy = self._place(state.data, place_rng, level)\n        data = mjx.forward(self.mjx_model, data)\n        info = dict(state.info)\n        info["curriculum"] = {"level": level, "origin": xy, "dist": jp.zeros(()), "cmd_dist": jp.zeros(()),\n                              "scan_offset": self._scan_offset(drift_rng),\n                              "fell": jp.zeros((), bool), "crossed": jp.zeros((), bool)}\n        metrics = dict(state.metrics, crossed=jp.zeros(()), terrain_level=level.astype(jp.float32))\n        obs = self._get_obs(data, info, self._contact(data))\n        return state.replace(data=data, obs=obs, info=info, metrics=metrics)\n\n    def _next_episode(self, state):\n        """First step of an episode: update the level from the last episode, then re-place."""\n        cur = state.info["curriculum"]\n        up = cur["dist"] > PROMOTE_DIST\n        down = ~up & (cur["fell"] | (cur["dist"] < DEMOTE_FRAC * cur["cmd_dist"]))\n        level = cur["level"] + up.astype(int) - down.astype(int)\n        info = dict(state.info)\n        info["rng"], lvl_rng, place_rng, drift_rng = jax.random.split(info["rng"], 4)\n        level = jp.where(level >= N_LEVELS, jax.random.randint(lvl_rng, (), 0, N_LEVELS), jp.maximum(level, 0))\n        data, xy = self._place(state.data, place_rng, level)\n        info["curriculum"] = {"level": level, "origin": xy, "dist": jp.zeros(()), "cmd_dist": jp.zeros(()),\n                              "scan_offset": self._scan_offset(drift_rng),\n                              "fell": jp.zeros((), bool), "crossed": jp.zeros((), bool)}\n        return state.replace(data=data, info=info)\n\n    def step(self, state, action):\n        new_episode = state.data.time == 0.0\n        placed = self._next_episode(state)\n        pick = lambda a, b: jp.where(new_episode, a, b)\n        # Only the keys _next_episode changes: info also holds the auto-reset wrapper\'s cached\n        # Data, whose MuJoCo Warp contact buffers are shared by the whole batch and must not be\n        # selected per env.\n        info = dict(state.info, rng=pick(placed.info["rng"], state.info["rng"]),\n                    curriculum=jax.tree.map(pick, placed.info["curriculum"], state.info["curriculum"]))\n        state = state.replace(\n            data=state.data.replace(qpos=pick(placed.data.qpos, state.data.qpos), qvel=pick(placed.data.qvel, state.data.qvel)),\n            info=info)\n        state = super().step(state, action)\n        info = dict(state.info)\n        cur = dict(info["curriculum"])\n        cur["dist"] = jp.linalg.norm(state.data.qpos[:2] - cur["origin"])\n        cur["cmd_dist"] = cur["cmd_dist"] + jp.linalg.norm(info["command"][:2]) * self.dt\n        cur["fell"] = self._fell(state.data)\n        crossed_now = (cur["dist"] > PROMOTE_DIST) & ~cur["crossed"]\n        cur["crossed"] = cur["crossed"] | crossed_now\n        info["curriculum"] = cur\n        metrics = dict(state.metrics, crossed=crossed_now.astype(jp.float32),\n                       terrain_level=cur["level"].astype(jp.float32))\n        return state.replace(info=info, metrics=metrics)\n\n    def _scan_offset(self, rng):\n        """Per-episode elevation-map offset (dx, dy, dz): odometry drift, camera model only."""\n        lim = jp.array([CAM_DRIFT_XY, CAM_DRIFT_XY, CAM_DRIFT_Z]) * self._config.noise_config.level\n        return jax.random.uniform(rng, (3,), minval=-lim, maxval=lim) * (self._config.scan_model == "camera")\n\n    def _camera_scan(self, data, info, rng):\n        """Height scan with the head depth camera\'s errors (see module docstring)."""\n        k_n, k_o, k_os, k_b = jax.random.split(rng, 4)\n        level = self._config.noise_config.level\n        off = info["curriculum"]["scan_offset"] if "curriculum" in info else jp.zeros(3)\n        base = data.qpos[:3] + jp.array([off[0], off[1], 0.0])\n        s = T.scan(self._hgrid, base, self._yaw(data), xp=jp) + off[2]\n        s = s + jax.random.normal(k_n, s.shape) * CAM_SIGMA * level\n        outlier = jax.random.bernoulli(k_o, CAM_OUTLIER_P * level, s.shape)\n        s = s + outlier * jax.random.normal(k_os, s.shape) * CAM_OUTLIER_SIGMA\n        blind_p = jp.asarray([CAM_BLIND_P.get(round(float(x), 2), 0.0) for x, _ in T.SCAN_PTS])\n        blind = jax.random.bernoulli(k_b, blind_p * level)\n        feet = jp.min(data.site_xpos[self._feet_site_id][:, 2]) - SOLE - (data.qpos[2] - T.NOMINAL_HEIGHT)\n        return jp.clip(jp.where(blind, feet, s), -1.0, 1.0)\n\n    def _get_obs(self, data, info, contact):\n        obs = super()._get_obs(data, info, contact)\n        scan = self._scan(data)\n        info["rng"], noise_rng = jax.random.split(info["rng"])\n        if self._config.scan_model == "camera":\n            noisy = self._camera_scan(data, info, noise_rng)\n        else:\n            noisy = scan + (2 * jax.random.uniform(noise_rng, scan.shape) - 1) * SCAN_NOISE * self._config.noise_config.level\n        return {\n            "state": jp.hstack([obs["state"], noisy]),\n            "privileged_state": jp.hstack([obs["privileged_state"], scan]),\n        }\n\n    def _fell(self, data):\n        up = self.get_gravity(data, "torso")[-1]   # torso z axis, world z component (1 = upright)\n        pelvis_rel = data.qpos[2] - T.lookup(self._hgrid, data.qpos[:2], xp=jp)\n        return (super()._get_termination(data) | (up < jp.cos(jp.deg2rad(MAX_TILT_DEG)))\n                | (pelvis_rel < MIN_PELVIS_REL))\n\n    def _get_termination(self, data):\n        off_map = jp.any(jp.abs(data.qpos[:2]) > T.HALF - 0.5)\n        return self._fell(data) | off_map\n\n    def _get_reward(self, data, action, info, metrics, done, first_contact, contact):\n        # the termination penalty is for falling, not for reaching the edge of the map\n        rewards = super()._get_reward(data, action, info, metrics, self._fell(data), first_contact, contact)\n        rewards["feet_edge"] = self._cost_feet_edge(data, contact)\n        pelvis_rel = data.qpos[2] - T.lookup(self._hgrid, data.qpos[:2], xp=jp)\n        rewards["base_height_rel"] = jp.square(jp.clip(BASE_HEIGHT_REL - pelvis_rel, 0.0, None))\n        return rewards\n\n    def _foot_axes(self, data):\n        """Foot positions (2, 3) and unit forward directions in the ground plane (2, 2)."""\n        feet = data.site_xpos[self._feet_site_id]\n        fwd = data.site_xmat[self._feet_site_id][:, :2, 0]\n        return feet, fwd / (jp.linalg.norm(fwd, axis=-1, keepdims=True) + 1e-6)\n\n    def _cost_feet_edge(self, data, contact):\n        """Loaded feet whose heel and toe sit on different steps (foot hanging over an edge)."""\n        feet, fwd = self._foot_axes(data)\n        heel = T.lookup(self._hgrid, feet[:, :2] + HEEL * fwd, xp=jp)\n        toe = T.lookup(self._hgrid, feet[:, :2] + TOE * fwd, xp=jp)\n        return jp.sum((jp.abs(toe - heel) > 0.02) * contact)\n\n    def _reward_feet_phase(self, data, phase, foot_height, command):\n        feet, fwd = self._foot_axes(data)\n        ground = T.lookup(self._hgrid, feet[:, :2], xp=jp)\n        for d in TOE_LOOKAHEAD:   # a riser just ahead raises the ground the swing foot must clear\n            ground = jp.maximum(ground, T.lookup(self._hgrid, feet[:, :2] + d * fwd, xp=jp))\n        foot_z = feet[..., -1] - ground\n        rz = gait.get_rz(phase, swing_height=foot_height)\n        reward = jp.exp(-jp.sum(jp.square(foot_z - rz)) / 0.01)\n        body_linvel = self.get_global_linvel(data, "pelvis")[:2]\n        body_angvel = self.get_global_angvel(data, "pelvis")[2]\n        moving = (jp.linalg.norm(body_linvel) > 0.1) | (jp.abs(body_angvel) > 0.1)\n        return reward * (moving | (jp.linalg.norm(command) > 0.01))\n', 'train.py': '"""Train the G1 step-length policy with Brax PPO (MuJoCo Playground recipe).\n\nRuns anywhere JAX runs: Kaggle/Colab GPU for real training, the Mac CPU for a\nsmoke test.\n\n    python -m g1pipe.train --timesteps 150_000_000 --out runs/steplength_v1      # GPU\n    python -m g1pipe.train --smoke --out runs/smoke                               # CPU check\n    python -m g1pipe.train --task stairs --timesteps 200_000_000 --out runs/stairs_v1   # stairs + height scan\n    python -m g1pipe.train --task stairs --init-from runs/g1-steplength-v1/run/params.pkl --out runs/stairs_v2\n                                                   # stairs, warm-started from the flat policy\n\nWrites to --out:  params.pkl (final), ckpt_*.pkl (periodic), progress.csv, config.json\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport functools\nimport json\nimport pickle\nimport time\nfrom pathlib import Path\n\nimport jax\nimport numpy as np\nfrom brax.training.agents.ppo import networks as ppo_networks\nfrom brax.training.agents.ppo import train as ppo\nfrom mujoco_playground import wrapper\nfrom mujoco_playground._src.locomotion.g1 import randomize as g1_randomize\nfrom mujoco_playground.config import locomotion_params\n\nfrom g1pipe.steplength_env import StepLength, default_config\n\n\ndef warm_start(path, obs_size):\n    """Brax restore_params from a trained policy whose observations are a prefix of ours.\n\n    The flat step-length policy\'s observations are the stairs policy\'s minus the trailing\n    height scan. New inputs get zero weights in the first layer (so the warm-started policy\n    starts out walking exactly like the flat one) and normaliser stats from the terrain.\n    """\n    from g1pipe import stairs_terrain as T\n    blob = pickle.load(open(path, "rb"))\n    norm, policy, value = blob["params"]\n    mean, std = T.scan_stats()\n    count = float(norm.count.to_numpy())\n    new = {"mean": dict(norm.mean), "std": dict(norm.std), "summed_variance": dict(norm.summed_variance)}\n    for key, (size,) in obs_size.items():\n        extra = size - blob["obs_size"][key][0]\n        if extra == 0:\n            continue\n        if extra != T.N_SCAN:\n            raise ValueError(f"{key}: {extra} new inputs, expected the {T.N_SCAN}-point height scan")\n        new["mean"][key] = np.concatenate([norm.mean[key], mean])\n        new["std"][key] = np.concatenate([norm.std[key], std])\n        new["summed_variance"][key] = np.concatenate([norm.summed_variance[key], std ** 2 * count])\n    norm = norm.replace(**new)\n\n    def pad_first_layer(p, extra):\n        k = p["params"]["hidden_0"]["kernel"]\n        p = jax.tree.map(lambda x: x, p)\n        p["params"]["hidden_0"]["kernel"] = np.concatenate([k, np.zeros((extra, k.shape[1]), k.dtype)])\n        return p\n\n    policy = pad_first_layer(policy, obs_size["state"][0] - blob["obs_size"]["state"][0])\n    value = pad_first_layer(value, obs_size["privileged_state"][0] - blob["obs_size"]["privileged_state"][0])\n    return norm, policy, value\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--out", default="runs/steplength")\n    ap.add_argument("--task", choices=["flat", "stairs"], default="flat",\n                    help="stairs: g1pipe.stairs_env (stair terrain + height scan observation)")\n    ap.add_argument("--timesteps", type=int, default=150_000_000)\n    ap.add_argument("--num-envs", type=int, default=None)\n    ap.add_argument("--impl", default=None, help="\'warp\' (NVIDIA GPU) or \'jax\'; default picks by backend")\n    ap.add_argument("--no-dr", action="store_true", help="disable domain randomisation (experiment E3)")\n    ap.add_argument("--step-scale", type=float, default=None, help="override step_length reward scale")\n    ap.add_argument("--scan-model", default=None, choices=["uniform", "camera"],\n                    help="stairs: height-scan noise; camera = head depth camera + elevation map errors")\n    ap.add_argument("--seed", type=int, default=0)\n    ap.add_argument("--init-from", default=None,\n                    help="params.pkl of a trained policy to start from (stairs: the flat step-length policy)")\n    ap.add_argument("--smoke", action="store_true", help="tiny CPU run to check the pipeline end to end")\n    a = ap.parse_args()\n\n    out = Path(a.out)\n    out.mkdir(parents=True, exist_ok=True)\n    backend = jax.default_backend()\n    print("JAX backend:", backend, "devices:", jax.devices())\n\n    if a.task == "stairs":\n        from g1pipe.stairs_env import StairsStepLength as Env, default_config as env_default_config\n    else:\n        Env, env_default_config = StepLength, default_config\n    env_cfg = env_default_config()\n    env_cfg.impl = a.impl or ("warp" if backend == "gpu" else "jax")\n    if a.step_scale is not None:\n        env_cfg.reward_config.scales.step_length = a.step_scale\n    if a.scan_model:\n        env_cfg.scan_model = a.scan_model\n    if a.no_dr:\n        env_cfg.push_config.enable = False\n        env_cfg.noise_config.level = 0.0\n\n    rl = locomotion_params.brax_ppo_config("G1JoystickFlatTerrain")\n    rl.num_timesteps = a.timesteps\n    if a.num_envs:\n        rl.num_envs = a.num_envs\n    if a.smoke:\n        rl.num_timesteps, rl.num_envs, rl.batch_size = 20_000, 32, 32\n        rl.num_minibatches, rl.num_evals, rl.episode_length = 4, 2, 100\n        rl.num_resets_per_eval = 0\n        env_cfg.naconmax, env_cfg.njmax = 64, env_cfg.njmax\n\n    env = Env(config=env_cfg)\n    if a.task == "stairs":\n        eval_env = Env(config=env_cfg, eval_levels=True)   # eval on every level, not the curriculum\'s\n        rl.num_resets_per_eval = 0                         # host resets would wipe curriculum levels\n    else:\n        eval_env = Env(config=env_cfg)\n    restore = warm_start(a.init_from, env.observation_size) if a.init_from else None\n\n    params_nf = dict(rl.network_factory)\n    train_kwargs = {k: v for k, v in rl.items() if k != "network_factory"}\n    network_factory = functools.partial(ppo_networks.make_ppo_networks, **params_nf)\n\n    (out / "config.json").write_text(json.dumps({\n        "env": env_cfg.to_dict(), "ppo": {**train_kwargs, "network_factory": params_nf},\n        "task": a.task, "no_dr": a.no_dr, "seed": a.seed, "backend": backend, "init_from": a.init_from,\n    }, indent=2, default=str))\n\n    t0 = time.time()\n    log = open(out / "progress.csv", "w", newline="")\n    writer = None\n\n    def progress(step, metrics):\n        nonlocal writer\n        row = {"step": step, "wall_s": round(time.time() - t0, 1),\n               **{k: float(v) for k, v in metrics.items() if k.startswith("eval/")}}\n        if writer is None:\n            writer = csv.DictWriter(log, fieldnames=list(row.keys()), extrasaction="ignore")\n            writer.writeheader()\n        writer.writerow(row)\n        log.flush()\n        print(f"[{row[\'wall_s\']:>7.0f}s] step {step:>11,}  reward {row.get(\'eval/episode_reward\', float(\'nan\')):8.2f}"\n              f"  step_len_err {row.get(\'eval/episode_step_len_err\', float(\'nan\')):.3f}"\n              + (f"  crossed {row[\'eval/episode_crossed\']:.2f}" if "eval/episode_crossed" in row else ""), flush=True)\n\n    def save_ckpt(step, make_policy, params):\n        with open(out / f"ckpt_{step:011d}.pkl", "wb") as f:\n            pickle.dump(jax.device_get(params), f)\n\n    train_fn = functools.partial(\n        ppo.train, **train_kwargs, network_factory=network_factory, seed=a.seed,\n        randomization_fn=None if a.no_dr else g1_randomize.domain_randomize,\n        progress_fn=progress, policy_params_fn=save_ckpt, restore_params=restore,\n    )\n    make_inference_fn, params, _ = train_fn(\n        environment=env, eval_env=eval_env, wrap_env_fn=wrapper.wrap_for_brax_training)\n\n    with open(out / "params.pkl", "wb") as f:\n        pickle.dump({"params": jax.device_get(params), "network_factory": params_nf,\n                     "obs_size": env.observation_size, "action_size": env.action_size, "task": a.task}, f)\n    print(f"done in {time.time() - t0:.0f}s -> {out}/params.pkl")\n\n\nif __name__ == "__main__":\n    main()\n'}
pkg = pathlib.Path('/content/src/g1pipe'); pkg.mkdir(parents=True, exist_ok=True)
for name, src in FILES.items():
    (pkg / name).write_text(src)
print('wrote', list(FILES))

In [ ]:
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = f'/content/drive/MyDrive/g1pipe_runs/{RUN_NAME}'
else:
    OUT = f'/content/runs/{RUN_NAME}'
os.makedirs(OUT, exist_ok=True); print(OUT)

In [ ]:
import subprocess, sys, os
env = dict(os.environ, PYTHONPATH='/content/src', XLA_PYTHON_CLIENT_MEM_FRACTION='0.9')
init = ['--init-from', f'/content/drive/MyDrive/{INIT_FROM}'] if (INIT_FROM and USE_DRIVE) else []
cmd = [sys.executable, '-u', '-m', 'g1pipe.train', '--out', OUT, '--timesteps', str(TIMESTEPS), *EXTRA_ARGS, *init]
p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if 'Warning' not in line and 'warn(' not in line:
        print(line, end='')
print('exit code', p.wait())

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv(f'{OUT}/progress.csv'); display(df.tail())
cols = [c for c in df.columns if c in ('eval/episode_reward', 'eval/episode_reward/step_length', 'eval/episode_crossed')]
df.plot(x='step', y=cols, subplots=True, figsize=(8, 6)); plt.show()

In [ ]:
# Download params + progress (checkpoints stay on Drive)
import shutil, tempfile, pathlib
tmp = pathlib.Path(tempfile.mkdtemp()) / RUN_NAME / 'run'
shutil.copytree(OUT, tmp, ignore=shutil.ignore_patterns('ckpt_*'))
zip_path = shutil.make_archive(f'/content/{RUN_NAME}', 'zip', tmp.parent.parent)
from google.colab import files as _f; _f.download(zip_path)